# 7.3 其他对齐方法

本节涵盖RLHF和DPO之外的其他对齐方法：RLAIF、KTO、ORPO、SimPO

## 1. RLAIF

**目的**：用AI反馈替代人类反馈

**基本原理**：使用强模型（如GPT-4）代替人类标注偏好数据，大幅降低标注成本。AI标注的偏好数据可用于训练奖励模型或直接用于DPO。

**优势**：成本远低于人工标注，可大规模生成偏好数据
**风险**：AI反馈可能存在系统性偏差

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

def ai_preference_scoring(prompt_emb, response_emb, quality_weight=0.7, safety_weight=0.3):
    quality_score = torch.cosine_similarity(prompt_emb, response_emb, dim=-1)
    safety_score = torch.sigmoid(response_emb.norm(dim=-1) - 1.0)
    return quality_weight * quality_score + safety_weight * safety_score

n_pairs = 8
d = 64
prompts = torch.randn(n_pairs, d)
chosen = prompts + 0.3 * torch.randn(n_pairs, d)
rejected = prompts + 0.8 * torch.randn(n_pairs, d)

scores_chosen = ai_preference_scoring(prompts, chosen)
scores_rejected = ai_preference_scoring(prompts, rejected)

print('=== RLAIF (RL from AI Feedback) ===')
print(f'AI scores - Chosen: {scores_chosen.tolist()}')
print(f'AI scores - Rejected: {scores_rejected.tolist()}')
consistency = (scores_chosen > scores_rejected).float().mean()
print(f'AI preference consistency: {consistency:.2%}')
print(f'\nKey: AI replaces human annotators for preference data generation.')
print(f'Much cheaper and more scalable than human annotation.')

## 2. KTO / ORPO / SimPO

**KTO（Kahneman-Tversky Optimization）**：
- 不需要成对偏好数据，只需要"好"或"不好"的标签
- 基于前景理论（Prospect Theory），损失对负例的惩罚大于对正例的奖励
- 数据获取更简单（不需要成对比较）

**ORPO（Odds Ratio Preference Optimization）**：
- 将SFT和对齐合并为一步
- 使用胜率比（odds ratio）作为偏好信号
- 无需参考模型，更高效

**SimPO（Simple Preference Optimization）**：
- 用序列的平均log概率代替参考模型的log概率
- 完全不需要参考模型
- 更简单、更高效

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimplePolicy(nn.Module):
    def __init__(self, d=64, v=50):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, v))
    def forward(self, x):
        return self.net(x)

policy = SimplePolicy()
x = torch.randn(8, 64)
labels = torch.randint(0, 50, (8,))
is_desirable = torch.tensor([1, 1, 0, 1, 0, 0, 1, 0])

logits = policy(x)
log_probs = F.log_softmax(logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)

z_ref = 0.0
beta_kto = 0.1
losses_kto = torch.where(
    is_desirable.bool(),
    1 - torch.sigmoid(beta_kto * (log_probs - z_ref)),
    1 - torch.sigmoid(beta_kto * (z_ref - log_probs))
)
kto_loss = losses_kto.mean()

logits_orpo = policy(x)
chosen_logits = logits_orpo + 0.5
rejected_logits = logits_orpo - 0.5
chosen_probs = F.softmax(chosen_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
rejected_probs = F.softmax(rejected_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
odds_ratio = (chosen_probs / (1 - chosen_probs + 1e-8)) / (rejected_probs / (1 - rejected_probs + 1e-8) + 1e-8)
orpo_loss = -F.logsigmoid(torch.log(odds_ratio + 1e-8)).mean()

avg_log_probs = log_probs
simpo_margin = 0.5
simpo_loss = -F.logsigmoid(beta_kto * (avg_log_probs[:4] - avg_log_probs[4:] - simpo_margin)).mean()

print('=== Alignment Methods Comparison ===')
print(f'\nKTO loss: {kto_loss.item():.4f}')
print(f'  -> Only needs desirable/undesirable labels (no pairs)')
print(f'  -> Based on Prospect Theory: asymmetric loss')
print(f'\nORPO loss: {orpo_loss.item():.4f}')
print(f'  -> Merges SFT + alignment in one step')
print(f'  -> Uses odds ratio as preference signal')
print(f'\nSimPO loss: {simpo_loss.item():.4f}')
print(f'  -> No reference model needed at all')
print(f'  -> Uses average log probability as implicit reward')
print(f'\nKey: Newer methods simplify alignment significantly.')
print(f'KTO: unpaired data. ORPO: one-step. SimPO: no reference model.')